In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, classification_report, 
    confusion_matrix, ConfusionMatrixDisplay
)
import xgboost as xgb
import matplotlib.pyplot as plt

# Load feature-engineered dataset
df = pd.read_csv('../data/processed/diabetic_data_features.csv')

# Separate features and target
X = df.drop(columns=['readmitted_30d'])
y = df['readmitted_30d']

print(f"Features: {X.shape}")
print(f"Target distribution:\n{y.value_counts(normalize=True).round(3)}")

Features: (69990, 81)
Target distribution:
readmitted_30d
0    0.91
1    0.09
Name: proportion, dtype: float64


In [2]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape} — positives: {y_train.mean().round(3)}")
print(f"Test:  {X_test.shape} — positives: {y_test.mean().round(3)}")

Train: (55992, 81) — positives: 0.09
Test:  (13998, 81) — positives: 0.09


In [7]:
# Calculate scale_pos_weight to handle class imbalance
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos

print(f"Negatives: {neg}, Positives: {pos}, Scale: {scale:.1f}")

# Train XGBoost
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='auc',
    early_stopping_rounds=20
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print("\nTraining complete.")

Negatives: 50964, Positives: 5028, Scale: 10.1
[0]	validation_0-auc:0.62764
[50]	validation_0-auc:0.64608
[92]	validation_0-auc:0.64807

Training complete.


Execute Evaluation

In [8]:
from sklearn.metrics import roc_auc_score, classification_report

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['No readmit', 'Readmit <30d']))

AUC-ROC: 0.6484

              precision    recall  f1-score   support

  No readmit       0.94      0.63      0.76     12741
Readmit <30d       0.13      0.57      0.22      1257

    accuracy                           0.63     13998
   macro avg       0.54      0.60      0.49     13998
weighted avg       0.87      0.63      0.71     13998



Precision — 0.13
De todos los pacientes que el modelo dijo "este va a ser readmitido", solo el 13% realmente lo fue. El 87% restante eran falsas alarmas — falsos positivos.
En español: el modelo es demasiado alarmista. Señala a muchos pacientes como de riesgo cuando en realidad no lo son.

Recall — 0.57
De los 1,257 pacientes que realmente fueron readmitidos, el modelo detectó el 57%. Los otros 43% los dejó pasar — falsos negativos. Eso es lo que tú ya identificaste como el problema grave.

Por qué probamos varios thresholds manualmente: queremos ver la tabla completa del trade-off antes de decidir cuál usar en producción.

In [9]:
from sklearn.metrics import precision_recall_curve

# Get precision, recall for every possible threshold
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

# Try a few specific thresholds
for t in [0.5, 0.4, 0.3, 0.2, 0.15, 0.1]:
    y_pred_t = (y_prob >= t).astype(int)
    recall_t = ((y_pred_t == 1) & (y_test == 1)).sum() / (y_test == 1).sum()
    precision_t = ((y_pred_t == 1) & (y_test == 1)).sum() / max((y_pred_t == 1).sum(), 1)
    print(f"Threshold {t}: Recall={recall_t:.3f}, Precision={precision_t:.3f}")

Threshold 0.5: Recall=0.572, Precision=0.133
Threshold 0.4: Recall=0.846, Precision=0.106
Threshold 0.3: Recall=0.989, Precision=0.092
Threshold 0.2: Recall=1.000, Precision=0.090
Threshold 0.15: Recall=1.000, Precision=0.090
Threshold 0.1: Recall=1.000, Precision=0.090


In [10]:
FINAL_THRESHOLD = 0.3

y_pred_final = (y_prob >= FINAL_THRESHOLD).astype(int)

print(classification_report(y_test, y_pred_final, target_names=['No readmit', 'Readmit <30d']))

              precision    recall  f1-score   support

  No readmit       0.97      0.03      0.07     12741
Readmit <30d       0.09      0.99      0.17      1257

    accuracy                           0.12     13998
   macro avg       0.53      0.51      0.12     13998
weighted avg       0.89      0.12      0.07     13998



In [12]:
# 1. Reentrenar el modelo final — parámetros definitivos
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='auc',
    early_stopping_rounds=20
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# 2. Recalcular probabilidades con ESTE modelo
y_prob = model.predict_proba(X_test)[:, 1]

print(y_prob[:10])
print(y_prob.min(), y_prob.max())

[0]	validation_0-auc:0.62764
[50]	validation_0-auc:0.64608
[92]	validation_0-auc:0.64807
[0.3470273  0.44620824 0.5059178  0.4325736  0.4446213  0.5792127
 0.54649454 0.59286326 0.4084745  0.40756115]
0.2109861 0.82398
